# Testing `OperationsData`

Builds the operations data with `create_operations_data()` and calls each method of the resulting object in its own cell, so the output can be explored step by step.

Run the cells from top to bottom. Every later cell uses the `ops` object created in the setup cell.

## Setup

In [9]:
from pathlib import Path
from pprint import pprint

from my_solution.load_data import create_operations_data

# The notebook lives in the project root, so the current folder is the project root
project_root = Path.cwd()

# Loads and validates the data, chunks and embeds the policies, and calculates the threshold
ops = create_operations_data(project_root)

## Attributes of the object

In [2]:
print("threshold:", ops.threshold)
print("top_k:", ops.top_k)
print("tickets:", len(ops.tickets))
print("customers:", len(ops.customers))
print("availability rows:", len(ops.availability))
print("chunks:", len(ops.document_chunks))

threshold: 0.41
top_k: 3
tickets: 7
customers: 7
availability rows: 4
chunks: 13


In [3]:
# Seed memory for follow-up turns
pprint(ops.memory)

{'conversation_id': 'northstar-demo',
 'last_customer_id': 'C-002',
 'last_ticket_id': 'T-1002',
 'notes': ['Previous turn verified Maya Chen as the requester for ticket '
           'T-1002.',
           'Maya asked for an MFA recovery checklist.']}


## `get_ticket`

In [9]:
ops.get_ticket("T-1001")

{'ticket_id': 'T-1001',
 'customer_id': 'C-001',
 'category': 'billing',
 'subject': 'Duplicate workshop payment',
 'body': 'I was charged twice for invoice INV-4482. Payment references are PAY-88A and PAY-88B. Can you refund the duplicate?'}

In [10]:
# Unknown ID: should raise ModelRetry
ops.get_ticket("T-9999")

ModelRetry: Unknown ticket_id: T-9999. Use an ID from the request or history, not a guess.

## `get_customer_context`

In [11]:
pprint(ops.get_customer_context("C-001"))

{'customer_id': 'C-001',
 'email': 'ari@example.com',
 'name': 'Ari Patel',
 'plan': 'Professional',
 'verified': True}


In [12]:
# Unknown ID: should raise ModelRetry
ops.get_customer_context("C-999")

ModelRetry: Unknown customer_id: C-999. Use an ID from the request or history, not a guess.

## `search_policy`

In [10]:
hits = ops.search_policy("I was charged twice and want a refund for the duplicate payment")
for hit in hits:
    print(hit["chunk_id"], round(hit["similarity"], 3))

refunds-0 0.709


In [11]:
hits

[{'chunk_id': 'refunds-0',
  'source_id': 'refunds',
  'title': 'Refunds And Duplicate Payments',
  'position': 0,
  'text': 'Refund requests must be checked against the original payment record before an\nagent promises a resolution. For duplicate payments, support should ask for the\ninvoice number, payment reference, and the last four digits of the payment\nmethod. If the duplicate charge is visible in the payment record, support may\ndraft a refund ticket update and mark the case for finance review.',
  'similarity': 0.7087438262960886}]

In [4]:
# Unsupported question: should return no chunks above the threshold
pprint(ops.search_policy("Where can I find next week's cafeteria menu?"))

ModelRetry: No relevant policy documents found for query: Where can I find next week's cafeteria menu?. Use a different query or check the knowledge base.

## `check_availability`

In [15]:
pprint(ops.check_availability("incident-response"))

{'available': True,
 'channel': '#northstar-incident-response',
 'owner': 'Iris Novak',
 'queue': 'incident-response'}


In [16]:
# Owner is not available
pprint(ops.check_availability("identity-review"))

{'available': False,
 'channel': '#northstar-identity-review',
 'owner': 'Nadine Keller',
 'queue': 'identity-review'}


In [17]:
# Unknown queue: should raise ModelRetry
ops.check_availability("billing-team")

ModelRetry: Unknown queue: billing-team. Use a queue from the availability list, not a guess.

## `draft_ticket_update`

In [18]:
print(ops.draft_ticket_update(
    ticket_id="T-1001",
    customer_name="Ari Patel",
    citations=["refunds-0"],
))

Draft update for Ari Patel on T-1001: we can proceed with the documented next step based on [refunds-0].


## `draft_escalation`

In [19]:
print(ops.draft_escalation(
    ticket_id="T-1003",
    queue="incident-response",
    owner="Iris Novak",
    citations=["incidents-0", "escalation-rules-0"],
))

Escalate T-1003 to incident-response (Iris Novak) with evidence [incidents-0], [escalation-rules-0].


In [20]:
# Owner not available: should raise an error
ops.draft_escalation(
    ticket_id="T-1007",
    queue="identity-review",
    owner="Nadine Keller",
    citations=["account-access-1"],
)

RuntimeError: No available owner for identity-review; request review.

In [21]:
# Wrong owner for the queue: should raise an error
ops.draft_escalation(
    ticket_id="T-1003",
    queue="incident-response",
    owner="Mateo Singh",
    citations=["incidents-0"],
)

RuntimeError: The proposed owner does not match incident-response; request review.

In [1]:
from my_solution.agent import build_agent, usage_limits

agent = build_agent()
limits = usage_limits()

/Users/emildietrich/github_repositories/mle-agent-project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1105 16:32:27.394000 98923 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 20363.58it/s]


In [2]:
import json
from pathlib import Path

cases = json.loads(Path("data/evaluation/agent_cases.json").read_text())

In [3]:
from my_solution.evaluation import case_to_prompt

case_1_prompt = case_to_prompt(cases[0])
print(case_1_prompt)

ticket_id: T-1001
prompt: Resolve the duplicate payment ticket and draft the next support update.



In [6]:
from pydantic_ai.exceptions import ModelHTTPError

try:
    response = await agent.run(case_1_prompt, usage_limits=limits)
    print(response.output.model_dump_json(indent=2))
except ModelHTTPError as error:
    print(f"ModelHTTPError: {error}")

{
  "answer": "Draft update for Ari Patel on T-1001: we can proceed with the documented next step based on [refunds-0].",
  "status": "answered",
  "source_ids": [
    "refunds"
  ],
  "review_required": false,
  "abstain_reason": null
}


In [5]:
from my_solution.evaluation import trace_from_result

trace = trace_from_result(
    response = response,
    case = cases[0]
)
for step in trace.tools:
    print(step)

PlanStep(tool='get_ticket', reason='Tool returned successfully.', arguments={'ticket_id': 'T-1001'})
PlanStep(tool='search_policy', reason='Tool returned successfully.', arguments={'query': 'duplicate payment refund policy'})
PlanStep(tool='get_customer_context', reason='Tool returned successfully.', arguments={'customer_id': 'C-001'})
PlanStep(tool='draft_ticket_update', reason='Tool returned successfully.', arguments={'citations': ['refunds-0'], 'customer_name': 'Ari Patel', 'ticket_id': 'T-1001'})
